# Pipeline B — LLM ordinal annotation and Mann-Whitney U

A thin wrapper over `adviceaudit` for exploring Pipeline B interactively.
The canonical way to run this analysis is `make llm`; this notebook calls the
same library functions so results are identical.

Set `MOCK = False` and export `ANTHROPIC_API_KEY` to use the real judge.
With `MOCK = True` the scores are deterministic placeholders and are
meaningless — they only demonstrate that the pipeline runs.

Run from the repository root.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from adviceaudit import mann_whitney
from adviceaudit.annotate import run as annotate_run
from adviceaudit.io_utils import load_config

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

INPUT = "../data/example/responses_example.csv"   # <-- change to your data
ANALYSIS_CONFIG = "../config/analysis.yaml"
RUBRIC_CONFIG = "../config/rubric.yaml"
MOCK = True                                        # <-- set False for real annotation

## 1. Annotate

Cached by (judge model, rubric hash, text), so re-running this cell is cheap and an interrupted run resumes.

In [ ]:
annotated = annotate_run(
    input_path=INPUT,
    output_path="../results/notebook/annotated.csv",
    analysis_config_path=ANALYSIS_CONFIG,
    rubric_config_path=RUBRIC_CONFIG,
    cache_path="../results/cache/annotations_notebook.jsonl",
    mock=MOCK,
)
dimensions = list(load_config(RUBRIC_CONFIG)["dimensions"])
annotated[["ID", "identity", *dimensions, "word_count"]].head()

In [ ]:
failed = annotated["annotation_failed"].sum()
if failed:
    print(f"WARNING: {failed} rows could not be annotated; their scores are blank.")
    print("Re-run the cell above to retry only those texts.")
else:
    print("All rows annotated successfully.")

## 2. Mann-Whitney U tests

In [ ]:
results = mann_whitney.run(
    input_path="../results/notebook/annotated.csv",
    output_path="../results/notebook/ordinal_results.csv",
    analysis_config_path=ANALYSIS_CONFIG,
    rubric_config_path=RUBRIC_CONFIG,
    distribution_output="../results/notebook/score_distribution.csv",
)
print(f"{len(results)} comparisons")
print(f"{int(results['meaningful_raw'].fillna(False).sum())} meaningful (raw p)")
print(f"{int(results['meaningful_adj'].fillna(False).sum())} meaningful (BH-adjusted p)")
results.head()

`effect_size` is the rank-biserial correlation, where **positive means
`identity_1` scored higher**. `meaningful_adj` applies the p < alpha and
|r| >= 0.3 rule to BH-adjusted p-values and is the stricter criterion to
report.

### Results table for one model and prompt

In [ ]:
def result_table(results, model, prompt):
    """Pivot the tidy results into one row per metric, styled for reading."""
    subset = results[
        (results["model"] == model) & (results["prompt"] == prompt)
    ].copy()
    if subset.empty:
        return None

    subset["pair"] = (
        subset["identity_1"].str[:3].str.upper() + " v " + subset["identity_2"].str[:3].str.upper()
    )
    table = subset.pivot(index="metric", columns="pair", values=["p_value_adj", "effect_size"])
    table.columns = [f"{pair} {'p(adj)' if stat == 'p_value_adj' else 'r'}"
                     for stat, pair in table.columns]
    return table.sort_index(axis=1).round(3)


table = result_table(results, model="model_a", prompt=1)

def highlight(val):
    if pd.isna(val):
        return ""
    return "font-weight: bold" if val < 0.05 else "color: grey"

if table is not None:
    display(
        table.style
        .map(highlight, subset=[c for c in table.columns if c.endswith("p(adj)")])
        .format(na_rep="---", precision=3)
        .set_caption("<b>Prompt 1 — model_a</b>")
    )

## 3. Score distributions

In [ ]:
distribution = pd.read_csv("../results/notebook/score_distribution.csv")
subset = distribution[(distribution["model"] == "model_a") & (distribution["prompt"] == 1)]
subset.pivot(index="metric", columns="identity", values="mean")

In [ ]:
from adviceaudit.plots import plot_score_distributions

columns = load_config(ANALYSIS_CONFIG)["columns"]
identities = load_config(ANALYSIS_CONFIG)["identities"]

paths = plot_score_distributions(
    annotated,
    columns=columns,
    metrics=dimensions,
    identities=identities,
    score_levels=[0, 1, 2, 3],
    output_dir="../results/notebook/figures",
)
print(f"Wrote {len(paths)} figures")

from IPython.display import Image
Image(str(paths[0]))